In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [2]:
# Common Imports (run once)

import os
import json
import time
import requests

from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, current_timestamp, lit, min, max, countDistinct, sum as _sum, when, sha2, concat_ws
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [3]:
API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

In [4]:
SILVER_PATH = "/home/jovyan/work/output/silver"

In [6]:
GOLD_PATH = "/home/jovyan/work/output/gold"

In [7]:
if not os.path.exists(GOLD_PATH):
    os.makedirs(GOLD_PATH)
    print("Gold folder created")
else:
    print("GOLD folder already exists")

GOLD folder already exists


In [8]:
# Load Silver
df_orders = spark.read.csv(f"{SILVER_PATH}/orders", header=True)
df_customers = spark.read.csv(f"{SILVER_PATH}/customers", header=True)
df_order_items = spark.read.csv(f"{SILVER_PATH}/order_items", header=True)

# Join orders + customers
df = df_orders.join(df_customers, "customer_id", "inner")

# Latest customer info (city/state)
window_latest = Window.partitionBy("customer_unique_id") \
                      .orderBy(col("order_purchase_timestamp").desc())

df_latest = df.withColumn("rn", row_number().over(window_latest)) \
              .filter(col("rn") == 1)

# Aggregations
df_agg = df \
    .groupBy("customer_unique_id") \
    .agg(
        min("order_purchase_timestamp").alias("first_order_date"),
        countDistinct("order_id").alias("total_orders")
    )

# Total spend
df_spend = df_order_items \
    .withColumn("total", col("price").cast("double") + col("freight_value").cast("double")) \
    .groupBy("order_id") \
    .agg(_sum("total").alias("order_total"))

df_spend = df_orders.join(df_spend, "order_id") \
    .groupBy("customer_id") \
    .agg(_sum("order_total").alias("total_spend"))

df_spend = df_spend.join(df_customers, "customer_id") \
    .groupBy("customer_unique_id") \
    .agg(_sum("total_spend").alias("total_spend"))

# Final join
df_final = df_latest \
    .join(df_agg, "customer_unique_id") \
    .join(df_spend, "customer_unique_id")

# Surrogate key
df_final = df_final.withColumn(
    "customer_key",
    sha2(concat_ws("||", col("customer_unique_id")), 256)
)

# Repeat flag
df_final = df_final.withColumn(
    "is_repeat_customer",
    when(col("total_orders") > 1, True).otherwise(False)
)

# Select final columns
df_final = df_final.select(
    "customer_key",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "first_order_date",
    "total_orders",
    "total_spend",
    "is_repeat_customer"
)

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{GOLD_PATH}/dim_customers")

print("dim_customers ready")

dim_customers ready


In [9]:
# Load Silver
df = spark.read.csv(f"{SILVER_PATH}/products", header=True)

# Type casting (important if not persisted)
df = df \
    .withColumn("product_weight_g", col("product_weight_g").cast("double")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("double")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("double")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("double")) \
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int"))

# Volume calculation
df = df.withColumn(
    "product_volume_cm3",
    when(
        col("product_length_cm").isNotNull() &
        col("product_height_cm").isNotNull() &
        col("product_width_cm").isNotNull(),
        col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
    )
)

# Surrogate key
df = df.withColumn(
    "product_key",
    sha2(concat_ws("||", col("product_id")), 256)
)

# Final columns
df_final = df.select(
    "product_key",
    "product_id",
    col("product_category_name"),
    "product_weight_g",
    "product_volume_cm3",
    "product_photos_qty",
    col("product_description_lenght").alias("product_description_length")
)

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{GOLD_PATH}/dim_products")

print("dim_products ready")

dim_products ready


In [10]:
# Load Silver
df = spark.read.csv(f"{SILVER_PATH}/sellers", header=True)

# Surrogate key
df = df.withColumn(
    "seller_key",
    sha2(concat_ws("||", col("seller_id")), 256)
)

# Final columns
df_final = df.select(
    "seller_key",
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
)

# Write
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{GOLD_PATH}/dim_sellers")

print("dim_sellers ready")

dim_sellers ready


In [11]:
from pyspark.sql.functions import col, sum as _sum, max as _max, row_number, when, datediff, sha2, concat_ws
from pyspark.sql.window import Window

# Load Silver
orders = spark.read.csv(f"{SILVER_PATH}/orders", header=True)
items = spark.read.csv(f"{SILVER_PATH}/order_items", header=True)
payments = spark.read.csv(f"{SILVER_PATH}/payments", header=True)
customers = spark.read.csv(f"{SILVER_PATH}/customers", header=True)

# Load Gold dimensions
dim_customers = spark.read.csv(f"{GOLD_PATH}/dim_customers", header=True)
dim_products = spark.read.csv(f"{GOLD_PATH}/dim_products", header=True)
dim_sellers = spark.read.csv(f"{GOLD_PATH}/dim_sellers", header=True)

# Type casting
orders = orders.withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp")) \
               .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp")) \
               .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp"))

items = items \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

payments = payments \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("payment_installments", col("payment_installments").cast("int"))

# -----------------------------------
# PAYMENT AGGREGATION
# -----------------------------------

# Total payment per order
payment_total = payments.groupBy("order_id") \
    .agg(_sum("payment_value").alias("total_payment"))

# Max installment
payment_installments = payments.groupBy("order_id") \
    .agg(_max("payment_installments").alias("payment_installments"))

# Payment type (highest value)
window_payment = Window.partitionBy("order_id") \
    .orderBy(col("payment_value").desc(), col("payment_type").asc())

payment_type = payments \
    .withColumn("rn", row_number().over(window_payment)) \
    .filter(col("rn") == 1) \
    .select("order_id", col("payment_type"))

# -----------------------------------
# ITEM LEVEL PREP
# -----------------------------------

items_total = items.groupBy("order_id") \
    .agg(_sum("price").alias("order_total_price"))

items = items.join(items_total, "order_id")

# -----------------------------------
# JOIN ALL
# -----------------------------------

df = items \
    .join(orders, "order_id") \
    .join(customers, "customer_id") \
    .join(payment_total, "order_id") \
    .join(payment_installments, "order_id") \
    .join(payment_type, "order_id")

# -----------------------------------
# DISTRIBUTE PAYMENT
# -----------------------------------

df = df.withColumn(
    "payment_value",
    (col("price") / col("order_total_price")) * col("total_payment")
)

# -----------------------------------
# DELIVERY METRICS
# -----------------------------------

df = df \
    .withColumn("days_to_deliver",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    ) \
    .withColumn("days_delivery_vs_estimate",
        datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))
    ) \
    .withColumn("is_late_delivery",
        when(col("days_delivery_vs_estimate") > 0, True)
        .when(col("days_delivery_vs_estimate").isNull(), None)
        .otherwise(False)
    )

# -----------------------------------
# JOIN DIMENSIONS
# -----------------------------------

# Join customer
df = df.join(dim_customers, "customer_unique_id", "left")

# Join product
df = df.join(dim_products, "product_id", "left")

# Join seller
df = df.join(dim_sellers, "seller_id", "left")

# -----------------------------------
# SURROGATE KEY
# -----------------------------------

df = df.withColumn(
    "order_item_sk",
    sha2(concat_ws("||", col("order_id"), col("order_item_id")), 256)
)

# -----------------------------------
# FINAL SELECT
# -----------------------------------

df_final = df.select(
    "order_item_sk",
    "order_id",
    col("order_item_id").cast("int"),
    "customer_key",
    "product_key",
    "seller_key",
    col("order_purchase_timestamp").alias("order_date"),
    "order_status",
    "price",
    "freight_value",
    "payment_value",
    "payment_type",
    "payment_installments",
    "days_to_deliver",
    "days_delivery_vs_estimate",
    "is_late_delivery"
)

# -----------------------------------
# WRITE
# -----------------------------------

df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{GOLD_PATH}/fact_order_items")

print("fact_order_items ready")

fact_order_items ready


In [12]:
fact_order_items = spark.read.csv(f"{GOLD_PATH}/fact_order_items", header=True)
dim_customers   = spark.read.csv(f"{GOLD_PATH}/dim_customers", header=True)
dim_products    = spark.read.csv(f"{GOLD_PATH}/dim_products", header=True)
dim_sellers     = spark.read.csv(f"{GOLD_PATH}/dim_sellers", header=True)

In [13]:
fact_order_items.createOrReplaceTempView("fact_order_items")
dim_customers.createOrReplaceTempView("dim_customers")
dim_products.createOrReplaceTempView("dim_products")
dim_sellers.createOrReplaceTempView("dim_sellers")

In [14]:
spark.sql("""
SELECT 
    order_status,
    COUNT(*) as total_orders,
    SUM(payment_value) as total_revenue
FROM fact_order_items
GROUP BY order_status
""").show()

+------------+------------+------------------+
|order_status|total_orders|     total_revenue|
+------------+------------+------------------+
|     shipped|         111|          16768.47|
|    canceled|          85|25279.409999999996|
|    invoiced|          38|           6777.09|
|   delivered|       14105| 2013318.139999994|
|  processing|           2|             79.07|
+------------+------------+------------------+



In [22]:
print("Fact count:", fact_order_items.count())
print("Customers:", dim_customers.count())
print("Products:", dim_products.count())
print("Sellers:", dim_sellers.count())

Fact count: 14341
Customers: 12585
Products: 32951
Sellers: 3095


In [16]:
spark.sql("""
SELECT SUM(price + freight_value) as total_sales
FROM fact_order_items
""").show()

+-----------------+
|      total_sales|
+-----------------+
|2062202.959999991|
+-----------------+



In [22]:
# 1st querRevenue Trend Analysis with Ranking
spark.sql("""
WITH base AS (
    SELECT 
        p.product_category_name,
        YEAR(f.order_date) AS year,
        MONTH(f.order_date) AS month,
        COUNT(*) AS transactions,
        SUM(f.price + f.freight_value) AS monthly_revenue
    FROM fact_order_items f
    JOIN dim_products p 
        ON f.product_key = p.product_key
    GROUP BY 
        p.product_category_name,
        YEAR(f.order_date),
        MONTH(f.order_date)
),

filtered AS (
    SELECT *
    FROM base
    WHERE transactions >= 10
),

top5 AS (
    SELECT product_category_name
    FROM filtered
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),

ranked AS (
    SELECT 
        f.*,
        RANK() OVER (
            PARTITION BY f.year, f.month 
            ORDER BY f.monthly_revenue DESC
        ) AS monthly_rank,
        
        LAG(f.monthly_revenue) OVER (
            PARTITION BY f.product_category_name 
            ORDER BY f.year, f.month
        ) AS prev_revenue
    FROM filtered f
    JOIN top5 t
        ON f.product_category_name = t.product_category_name
)

SELECT 
    product_category_name,
    year,
    month,
    monthly_revenue,
    monthly_rank,

    CASE 
        WHEN prev_revenue IS NULL OR prev_revenue = 0 THEN NULL
        ELSE (monthly_revenue - prev_revenue) / prev_revenue
    END AS mom_growth_pct,

    AVG(monthly_revenue) OVER (
        PARTITION BY product_category_name 
        ORDER BY year, month 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3m_avg_revenue

FROM ranked
ORDER BY product_category_name, year, month
""").show()

+---------------------+----+-----+------------------+------------+--------------------+----------------------+
|product_category_name|year|month|   monthly_revenue|monthly_rank|      mom_growth_pct|rolling_3m_avg_revenue|
+---------------------+----+-----+------------------+------------+--------------------+----------------------+
|         beleza_saude|2018|    7|122743.98000000003|           1|                NULL|    122743.98000000003|
|         beleza_saude|2018|    8|137206.07000000004|           1|  0.1178232121852331|    129975.02500000002|
|      cama_mesa_banho|2018|    7| 68708.37999999996|           4|                NULL|     68708.37999999996|
|      cama_mesa_banho|2018|    8| 75207.83000000005|           3| 0.09459472046932395|     71958.10500000001|
|        esporte_lazer|2018|    7| 65830.02000000003|           5|                NULL|     65830.02000000003|
|        esporte_lazer|2018|    8| 60374.74999999999|           5| -0.0828690314844206|     63102.38500000001|
|

In [23]:
# 2nd query_Seller Performance Scorecard
spark.sql("""
WITH base AS (
    SELECT 
        s.seller_id,
        s.seller_state,
        COUNT(*) AS total_orders,
        SUM(f.price + f.freight_value) AS total_revenue,
        
        -- Late delivery rate
        SUM(CASE WHEN f.is_late_delivery = true THEN 1 ELSE 0 END) * 1.0 / COUNT(*) 
            AS late_delivery_rate,
        
        -- Avg delivery deviation
        AVG(f.days_delivery_vs_estimate) AS avg_days_vs_estimate

    FROM fact_order_items f
    JOIN dim_sellers s
        ON f.seller_key = s.seller_key   

    GROUP BY s.seller_id, s.seller_state
),

filtered AS (
    SELECT *
    FROM base
    WHERE total_orders >= 20
),

scored AS (
    SELECT *,
        
        1 - PERCENT_RANK() OVER (ORDER BY late_delivery_rate) 
            AS on_time_pctl,
        
        1 - PERCENT_RANK() OVER (ORDER BY avg_days_vs_estimate) 
            AS speed_pctl,
        
        PERCENT_RANK() OVER (ORDER BY total_revenue) 
            AS revenue_pctl

    FROM filtered
),

final AS (
    SELECT *,
        (0.4 * on_time_pctl +
         0.3 * speed_pctl +
         0.3 * revenue_pctl) AS composite_score
    FROM scored
)

SELECT 
    seller_id,
    seller_state,
    total_orders,
    total_revenue,
    late_delivery_rate,
    avg_days_vs_estimate,
    on_time_pctl,
    speed_pctl,
    revenue_pctl,
    composite_score,

    RANK() OVER (ORDER BY composite_score DESC) AS overall_rank

FROM final
ORDER BY overall_rank
""").show()

+--------------------+------------+------------+------------------+------------------+--------------------+------------------+------------------+-------------------+------------------+------------+
|           seller_id|seller_state|total_orders|     total_revenue|late_delivery_rate|avg_days_vs_estimate|      on_time_pctl|        speed_pctl|       revenue_pctl|   composite_score|overall_rank|
+--------------------+------------+------------+------------------+------------------+--------------------+------------------+------------------+-------------------+------------------+------------+
|3d871de0142ce09b7...|          SP|          59|           7108.76|0.0000000000000000| -14.728813559322035|               1.0|0.9675324675324676| 0.7532467532467533|0.9162337662337663|           1|
|58f1a6197ed863543...|          MG|          36|           6382.24|0.0000000000000000| -15.294117647058824|               1.0|0.9805194805194806| 0.7337662337662337|0.9142857142857144|           2|
|de722cd6d